In [ ]:
import json
import pathlib

import pandas as pd

In [ ]:
from kebab.utils.dataset.wikidata.wikidata_type_hierarchy_extractor import WikidataTypeHierarchyExtractor

Sample element from the dataset
---

In [ ]:
# load the Wikidata hierarchy
hierarchy_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "Wikidata"
    / "Type Hierarchy"
    / "2025-01-30"
    / "wikidata_type_hierarchy.jsonl"
)

graph = {}
with open(hierarchy_path, encoding="utf-8") as f:
    for line in f:
        node = json.loads(line.strip())
        graph[node["id"]] = node

print(f"Type hierarchy contains {len(graph):,d} nodes")
print("Sample node:")
graph["Q5"]

Main root node (Entity)
---

In [ ]:
graph["Q35120"]

Nodes with the most references
---


In [ ]:
# nodes with the most references
df = pd.DataFrame(
    [(node["id"], node["name"], node["ref_count"], node["descriptions"]) for node in graph.values()],
    columns=["id", "name", "ref_count", "descriptions"],
)

df = df.sort_values("ref_count", ascending=False).reset_index(drop=True)
df.head(10)

Nodes with the highest merge number
---

These are nodes that have been merged with other nodes due to same-to-be-the-same-as links or parent relationship cycles    

In [ ]:
# nodes with largest merged node count
df = pd.DataFrame(
    [
        (node["id"], node["name"], node["aliases"], len(node["merged_ids"]), node["ref_count"])
        for node in graph.values()
    ],
    columns=["id", "name", "aliases", "merged_count", "ref_count"],
)

df = df.sort_values("merged_count", ascending=False).reset_index(drop=True)
df.head(10)

Root nodes with the highest combined reference count
---

These are the top-level nodes in the hierarchy that have the most references when considering all their children nodes.

In [ ]:
# largest root nodes
root_node_ids = WikidataTypeHierarchyExtractor.get_root_entities(graph)
print(f"Root nodes: {len(root_node_ids):,d}")

root_nodes = [graph[node_id] for node_id in root_node_ids]

df = pd.DataFrame(
    [(node["id"], node["name"], node["subgraph_ref_count"], node["descriptions"]) for node in root_nodes],
    columns=["id", "name", "subgraph_ref_count", "descriptions"],
)

df = df.sort_values("subgraph_ref_count", ascending=False).reset_index(drop=True)
df.head(20)

Leaf nodes with the highest reference count
---

In [ ]:
# largest leaf nodes
leaf_node_ids = WikidataTypeHierarchyExtractor.get_leaf_entities(graph)
print(f"Leaf nodes: {len(leaf_node_ids):,d}")

leaf_nodes = [graph[node_id] for node_id in leaf_node_ids]
df = pd.DataFrame(
    [(node["id"], node["name"], node["subgraph_ref_count"], node["descriptions"]) for node in leaf_nodes],
    columns=["id", "name", "subgraph_ref_count", "descriptions"],
)

df = df.sort_values("subgraph_ref_count", ascending=False).reset_index(drop=True)
df.head(20)